# Validación Cruzada Temporal
## Walk-forward, expanding y sliding

> **Fase 4 — Video 19** | Evaluación, Validación y Producción
> Dataset: Histórico de Ventas Sintético

---

### El error #1 que hunde proyectos en producción

Validar una serie de tiempo **como si fuera un problema tabular** — con `KFold` aleatorio — es el error más
caro del forecasting. Al barajar las filas, metes el **futuro** en el entrenamiento: el modelo "ve" lo que
debería predecir, las métricas salen preciosas… y en producción todo se derrumba.

La regla es simple y sagrada: **nunca entrenes con datos posteriores a los que evalúas.** En este video
implementamos la validación que respeta el tiempo y descubrimos algo incómodo: **un solo hold-out puede
mentirte**.

### Ruta del notebook
1. Librerías y carga
2. El error mortal: KFold aleatorio (fuga de datos)
3. Los tres esquemas temporales
4. Walk-forward: un solo número miente, la distribución no
5. Expanding vs sliding: cuál elegir
6. Conclusiones y puente al Video 20

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.backtest import temporal_splits, walk_forward
from src.metrics import mae, wape

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
y = df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]
yv = y.values.astype(float)
print(f'✅ Serie semanal: {len(y)} semanas')

---
## 2. El error mortal: KFold aleatorio

`KFold(shuffle=True)` reparte las semanas **al azar** entre train y test. En una serie con memoria, eso
significa entrenar con semanas *posteriores* a las que evalúas — **fuga de datos**. El mismo modelo, con la
validación correcta (temporal), reporta un error mayor: el verdadero.

In [ ]:
d = pd.DataFrame({'y': yv,
                  'lag1': pd.Series(yv).shift(1),
                  'lag52': pd.Series(yv).shift(52)}).dropna()
X, t = d[['lag1', 'lag52']].values, d['y'].values

# ❌ KFold aleatorio (baraja el tiempo)
errs_random = []
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    m = LinearRegression().fit(X[tr], t[tr])
    errs_random.append(mae(t[te], m.predict(X[te])))

# ✅ Validación temporal
errs_temporal = []
for tr, te in temporal_splits(len(t), initial=104, horizon=13, step=13):
    m = LinearRegression().fit(X[tr], t[tr])
    errs_temporal.append(mae(t[te], m.predict(X[te])))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(['KFold aleatorio\n(con fuga)', 'CV temporal\n(honesta)'],
       [np.mean(errs_random), np.mean(errs_temporal)], color=[RED, GREEN], width=0.5)
ax.set_ylabel('MAE')
ax.set_title('El KFold aleatorio subestima el error', fontsize=13, fontweight='bold')
for i, v in enumerate([np.mean(errs_random), np.mean(errs_temporal)]):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

opt = 1 - np.mean(errs_random) / np.mean(errs_temporal)
print(f'KFold aleatorio: MAE {np.mean(errs_random):,.0f}  (parece {opt:.0%} mejor… es mentira)')
print(f'CV temporal    : MAE {np.mean(errs_temporal):,.0f}  (el error real)')
print('El optimismo crece con la tendencia y la autocorrelación. En series con memoria,')
print('el KFold aleatorio SIEMPRE te engaña. Nunca lo uses para series de tiempo.')

---
## 3. Los tres esquemas temporales

Todos evalúan sobre el **futuro** de cada punto de corte. Se diferencian en la ventana de entrenamiento:

| Esquema | Ventana de train | Cuándo |
|---|---|---|
| **Hold-out simple** | Una sola partición al final | Rápido, pero frágil (un solo número) |
| **Expanding** | Crece: usa *toda* la historia hasta el corte | Series estables; aprovecha todo el dato |
| **Sliding** | Tamaño fijo que se desliza | Series con cambios de régimen; olvida lo viejo |

**Walk-forward** = avanzar el origen fold a fold y evaluar en cada uno. Visualicémoslo.

In [ ]:
def plot_scheme(ax, folds, title):
    for i, (tr, te) in enumerate(folds):
        ax.barh(i, len(tr), left=tr[0], color=BLUE, height=0.6)
        ax.barh(i, len(te), left=te[0], color=ORANGE, height=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('semana'); ax.set_ylabel('fold'); ax.invert_yaxis()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
plot_scheme(axes[0], temporal_splits(len(yv), 104, 13, 13, 'expanding'), 'Expanding (train crece)')
plot_scheme(axes[1], temporal_splits(len(yv), 104, 13, 13, 'sliding'), 'Sliding (ventana fija)')
handles = [plt.Rectangle((0,0),1,1,color=BLUE), plt.Rectangle((0,0),1,1,color=ORANGE)]
fig.legend(handles, ['train', 'test'], loc='upper right')
fig.suptitle('Walk-forward: el origen avanza fold a fold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Azul = entrenamiento, naranja = evaluación. El test SIEMPRE va después del train.')

---
## 4. Walk-forward: un solo número miente, la distribución no

Evaluamos un **seasonal naive** en 8 orígenes distintos (trimestres). En vez de un WAPE, obtenemos ocho —
y su dispersión es la verdad: dependiendo del trimestre que te toque de hold-out, reportarías cosas muy
distintas.

In [ ]:
snaive = lambda tr, h: tr[-52:][:h]        # seasonal naive
wapes = [wape(r, p) for r, p in walk_forward(yv, snaive, initial=104, horizon=13, step=13)]

fig, ax = plt.subplots(figsize=(12, 4.5))
folds = range(1, len(wapes) + 1)
ax.bar(folds, wapes, color=BLUE, alpha=0.85)
ax.axhline(np.mean(wapes), color=RED, linewidth=2, label=f'media {np.mean(wapes):.1%}')
ax.axhline(wapes[-1], color=ORANGE, linestyle='--', linewidth=1.5,
           label=f'si solo usaras el ÚLTIMO fold: {wapes[-1]:.1%}')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('fold (origen temporal)'); ax.set_ylabel('WAPE')
ax.set_title('WAPE del seasonal naive en 8 orígenes: la dispersión es la verdad',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'WAPE por fold: {[f"{w:.1%}" for w in wapes]}')
print(f'  media {np.mean(wapes):.1%} | rango {min(wapes):.1%}–{max(wapes):.1%} | std {np.std(wapes):.1%}')
print(f'\nUn hold-out único podría reportar desde {min(wapes):.0%} hasta {max(wapes):.0%}: ¡el doble!')
print('Reporta SIEMPRE media ± dispersión sobre varios folds, no un número suelto y con suerte.')

---
## 5. Expanding vs sliding: ¿cuál elegir?

Con un modelo que usa **toda** la ventana de entrenamiento (aquí, un seasonal naive con deriva estimada en
la ventana), los dos esquemas ya difieren. La elección depende de tu serie.

In [ ]:
def drift_snaive(tr, h, m=52):
    shape = tr[-m:][:h]
    slope = np.polyfit(np.arange(len(tr)), tr, 1)[0]
    return shape + slope * np.arange(1, h + 1)

res = {}
for mode in ('expanding', 'sliding'):
    res[mode] = [wape(r, p) for r, p in
                 walk_forward(yv, drift_snaive, initial=104, horizon=13, step=13, mode=mode)]

fig, ax = plt.subplots(figsize=(12, 4.5))
x = np.arange(1, len(res['expanding']) + 1)
ax.plot(x, res['expanding'], marker='o', color=BLUE, label=f"expanding (media {np.mean(res['expanding']):.1%})")
ax.plot(x, res['sliding'], marker='s', color=ORANGE, label=f"sliding (media {np.mean(res['sliding']):.1%})")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('fold'); ax.set_ylabel('WAPE')
ax.set_title('Expanding vs Sliding (mismo modelo con deriva)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Expanding: media WAPE {np.mean(res['expanding']):.3f}")
print(f"Sliding  : media WAPE {np.mean(res['sliding']):.3f}")
print('\nGuía práctica:')
print('  • Expanding: series estables → aprovecha toda la historia (aquí gana ligeramente).')
print('  • Sliding: si hubo un cambio de régimen (nuevo producto, pandemia, cambio de precios),')
print('    olvidar lo viejo evita arrastrar patrones muertos.')

---
## 6. Conclusiones y puente al Video 20

### Las reglas que te llevas

1. **Nunca uses KFold aleatorio en series de tiempo.** Baraja el tiempo, mete el futuro en el train y te
   da métricas mentirosas.
2. **Valida en el futuro:** el test siempre después del train. Punto.
3. **Un solo hold-out miente.** Usa **walk-forward** con varios orígenes y reporta **media ± dispersión** —
   la variabilidad entre folds es enorme.
4. **Expanding** para series estables (usa todo el dato); **sliding** para series con cambios de régimen.
5. **La validación honesta es la base de todo lo demás** — incluida la optimización de hiperparámetros.

### El puente

Ahora que sabemos validar sin engañarnos, podemos **optimizar** hiperparámetros… sin sobreajustar al
pasado. Acoplar la búsqueda con la validación temporal correcta es lo que separa un modelo robusto de uno
que memoriza.

---

### Próximo video
**Video 20 — Optimización de hiperparámetros con Optuna (sin sobre-ajustar al pasado)**
Optuna + validación temporal, bien acopladas: pruning, early stopping y cómo no caer en p-hacking.